In [3]:
import numpy as np
from common.time_layers import *

In [2]:
class RNN:

    def __init__(self, Wx, Wh, b):
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.cache = None

    def forward(self, x, prev_h):
        Wx, Wh, b = self.params
        a = np.dot(prev_h, Wh) + np.dot(x, Wh) + b
        next_h = np.tanh(a)
        self.cache = (x, prev_h, next_h)
        return next_h

    def backward(self, dh_next):
        x, prev_h, next_h = self.cache
        Wx, Wh, b = self.params

        dt = dh_next * (1 - next_h ** 2)
        db = np.sum(dt, axis=0)
        dWh = np.dot(prev_h.T, dt)
        dh_prev = np.dot(dt, Wh.T)
        dWx = np.dot(x.T, dt)
        dx = np.dot(dt, Wx.T)

        self.grads[0][...] = dWx
        self.grads[1][...] = dWh
        self.grads[2][...] = db

        return dx, dh_prev



In [ ]:
class TimeRNN:
    def __init__(self, Wx, Wh, b, stateFul = False):
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.layers = []
        self.stateFul = stateFul
        self.h, self.dh = None, None

    def set_state(self, h):
        self.h = h
    def reset_state(self):
        self.h = None


    def forward(self, inputs):
        Wx, Wh, b = self.params
        batch_size, time_steps, num_inputs = inputs.shape
        num_inputs, hidden_nums = Wx.shape

        self.layers = []
        hs = np.empty((batch_size, time_steps, hidden_nums), dtype= 'f')

        if not self.stateFul or self.h is None:
            self.h = np.zeros((batch_size, hidden_nums), dtype = 'f')

        for time_step in range(time_steps):
            layer = RNN(*self.params)
            self.h = layer.forward(inputs[:,time_step,:], self.h)
            hs[:,time_step,:] = self.h
            self.layers.append(layer)
        return hs

    def backward(self, dhs):
        Wx, Wh, b = self.params


